# LAB-3: Navegação Autônoma com Evitação de Obstáculos

Neste laboratório, implementaremos:
- **Algoritmo de evitação reativa** (Bug Algorithm)
- **Navegação autônoma** para um alvo
- **Trajetória do robô** visualizada
- **Estados de navegação** (buscando, contornando, atingido)

In [ ]:
# CÓDIGO LAB-3: Navegação Autônoma com Evitação de Obstáculos
import pygame
import math
import numpy as np
from collections import deque

LARGURA, ALTURA = 1000, 750
FPS = 60
COR_FUNDO = (20, 24, 30)
COR_ROBO = (0, 200, 255)
COR_OBSTACULO = (180, 50, 50)
COR_RAIO_LIVRE = (0, 255, 100)
COR_RAIO_COLISAO = (255, 200, 0)
COR_ALVO = (255, 100, 100)
COR_TRAJETORIA = (100, 150, 200)

class AutonomousRobot:
    """Robô com capacidade de navegação autônoma."""
    
    def __init__(self, x, y, theta=0.0):
        self.x = float(x)
        self.y = float(y)
        self.theta = float(theta)
        
        # Sensores
        self.num_sensores = 8
        self.sensor_angles = [2 * math.pi * i / self.num_sensores for i in range(self.num_sensores)]
        self.sensor_range = 150.0
        self.sensor_readings = [self.sensor_range] * self.num_sensores
        
        # Navegação
        self.target_x = None
        self.target_y = None
        self.state = "idle"  # idle, navigating, obstacle_avoidance, reached
        self.trajetoria = deque(maxlen=500)
        self.velocidade = 0.0
        self.velocidade_max = 3.0
        self.aceleracao = 0.1
        
        # Parâmetros de controle
        self.k_alvo = 0.02  # Ganho para direção do alvo
        self.k_obstaculo = 0.05  # Ganho para evitação
        self.distancia_min = 50  # Distância mínima de segurança
        self.tolerancia_alvo = 15  # Distância para considerar alvo atingido

    def cast_rays(self, obstacles):
        """Verifica interseção dos raios."""
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            dx = math.cos(angle)
            dy = math.sin(angle)
            
            min_distance = self.sensor_range
            
            for obs in obstacles:
                distance = self._ray_rect_intersection(self.x, self.y, dx, dy, obs)
                if distance is not None and distance < min_distance:
                    min_distance = distance
            
            self.sensor_readings.append(min_distance)

    def _ray_rect_intersection(self, x0, y0, dx, dy, rect):
        """Calcula interseção ray-rectangle."""
        rx, ry, rw, rh = rect
        x_min, x_max = rx, rx + rw
        y_min, y_max = ry, ry + rh
        
        t_min = float('-inf')
        t_max = float('inf')
        
        if abs(dx) > 1e-6:
            t1 = (x_min - x0) / dx
            t2 = (x_max - x0) / dx
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif x0 < x_min or x0 > x_max:
            return None
        
        if abs(dy) > 1e-6:
            t1 = (y_min - y0) / dy
            t2 = (y_max - y0) / dy
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif y0 < y_min or y0 > y_max:
            return None
        
        if t_min < t_max and t_min > 0 and t_min < self.sensor_range:
            return t_min
        
        return None

    def set_target(self, x, y):
        """Define alvo para navegação."""
        self.target_x = x
        self.target_y = y
        self.state = "navigating"

    def compute_control(self, obstacles):
        """Computa velocidade linear e angular para evitar obstáculos."""
        # Verifica distância mínima entre sensores
        min_sensor = min(self.sensor_readings)
        
        # Calcula vetor de repulsão (evitação)
        repulsion_x = 0
        repulsion_y = 0
        
        for i, distance in enumerate(self.sensor_readings):
            if distance < self.distancia_min:
                angle = self.theta + self.sensor_angles[i]
                # Força inversamente proporcional à distância
                forca = (self.distancia_min - distance) / self.distancia_min
                repulsion_x -= forca * math.cos(angle)
                repulsion_y -= forca * math.sin(angle)
        
        # Calcula vetor de atração (direção do alvo)
        if self.target_x is not None and self.target_y is not None:
            dx_alvo = self.target_x - self.x
            dy_alvo = self.target_y - self.y
            dist_alvo = math.sqrt(dx_alvo**2 + dy_alvo**2)
            
            if dist_alvo < self.tolerancia_alvo:
                self.state = "reached"
                self.velocidade = 0
                return 0, 0
            
            # Normaliza
            if dist_alvo > 0:
                dx_alvo /= dist_alvo
                dy_alvo /= dist_alvo
            
            # Combina atração e repulsão
            direcao_x = dx_alvo + repulsion_x * self.k_obstaculo
            direcao_y = dy_alvo + repulsion_y * self.k_obstaculo
            
            # Calcula ângulo desejado
            angle_desejado = math.atan2(direcao_y, direcao_x)
            
            # Diferença angular
            delta_theta = angle_desejado - self.theta
            
            # Normaliza entre -pi e pi
            while delta_theta > math.pi:
                delta_theta -= 2 * math.pi
            while delta_theta < -math.pi:
                delta_theta += 2 * math.pi
            
            # Controle angular
            velocidade_angular = 0.05 * delta_theta
            
            # Controle linear (reduz velocidade se há obstáculo)
            if min_sensor < self.distancia_min:
                self.velocidade = max(0, self.velocidade - self.aceleracao)
                self.state = "obstacle_avoidance"
            else:
                self.velocidade = min(self.velocidade_max, self.velocidade + self.aceleracao)
                self.state = "navigating"
            
            return self.velocidade, velocidade_angular
        
        return 0, 0

    def update(self, obstacles):
        """Atualiza posição do robô."""
        self.cast_rays(obstacles)
        
        velocidade, velocidade_angular = self.compute_control(obstacles)
        
        # Atualiza posição
        self.x += velocidade * math.cos(self.theta)
        self.y += velocidade * math.sin(self.theta)
        self.theta += velocidade_angular
        
        # Limita dentro da tela
        self.x = max(0, min(LARGURA, self.x))
        self.y = max(0, min(ALTURA, self.y))
        
        # Armazena trajetória
        self.trajetoria.append((self.x, self.y))

    def draw(self, screen):
        """Desenha o robô e trajetória."""
        # Trajetória
        if len(self.trajetoria) > 1:
            pygame.draw.lines(screen, COR_TRAJETORIA, list(self.trajetoria), 1)
        
        # Corpo do robô
        pygame.draw.circle(screen, COR_ROBO, (int(self.x), int(self.y)), 8)
        pygame.draw.line(screen, COR_ROBO, (self.x, self.y),
                         (self.x + 12 * math.cos(self.theta),
                          self.y + 12 * math.sin(self.theta)), 2)
        
        # Raios dos sensores
        for i, beta in enumerate(self.sensor_angles):
            angle = self.theta + beta
            distance = self.sensor_readings[i]
            end_x = self.x + distance * math.cos(angle)
            end_y = self.y + distance * math.sin(angle)
            
            cor = COR_RAIO_COLISAO if distance < self.sensor_range - 0.1 else COR_RAIO_LIVRE
            pygame.draw.line(screen, cor, (self.x, self.y), (end_x, end_y), 1)


def draw_obstacles(screen, obstacles):
    """Desenha obstáculos."""
    for obs in obstacles:
        x, y, w, h = obs
        pygame.draw.rect(screen, COR_OBSTACULO, (x, y, w, h))


def draw_alvo(screen, x, y, raio=10):
    """Desenha o alvo."""
    if x is not None and y is not None:
        pygame.draw.circle(screen, COR_ALVO, (int(x), int(y)), raio)
        pygame.draw.circle(screen, COR_ALVO, (int(x), int(y)), raio - 2, 1)


def main():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB-3: Navegação Autônoma com Evitação")
    clock = pygame.time.Clock()
    
    obstacles = [
        (100, 100, 150, 30),
        (400, 150, 30, 200),
        (650, 400, 150, 30),
        (300, 450, 250, 30),
        (50, 350, 30, 150),
    ]
    
    robot = AutonomousRobot(100, 100)
    robot.set_target(850, 100)
    
    running = True
    while running:
        clock.tick(FPS)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
            elif event.type == pygame.MOUSEBUTTONDOWN:
                # Click do mouse define novo alvo
                x, y = pygame.mouse.get_pos()
                robot.set_target(x, y)
        
        robot.update(obstacles)
        
        # Renderiza
        screen.fill(COR_FUNDO)
        draw_obstacles(screen, obstacles)
        robot.draw(screen)
        draw_alvo(screen, robot.target_x, robot.target_y)
        
        # HUD
        font = pygame.font.Font(None, 20)
        
        info_texts = [
            f"Posição: ({robot.x:.0f}, {robot.y:.0f})",
            f"Velocidade: {robot.velocidade:.2f}",
            f"Estado: {robot.state.upper()}",
            f"Distância até alvo: {math.sqrt((robot.target_x - robot.x)**2 + (robot.target_y - robot.y)**2) if robot.target_x else 'N/A':.0f}",
        ]
        
        for i, text in enumerate(info_texts):
            surface = font.render(text, True, (200, 200, 200))
            screen.blit(surface, (10, 10 + i * 25))
        
        inst_font = pygame.font.Font(None, 16)
        inst = inst_font.render("Click do mouse: Define novo alvo | ESC: Sair", True, (150, 150, 150))
        screen.blit(inst, (10, ALTURA - 25))
        
        pygame.display.flip()
    
    pygame.quit()

if __name__ == "__main__":
    main()